# SoftMoE-Retrieval — Train RIÊNG từng bộ DML
Hybrid CNN+ViT · Multi-scale TokenLearner · Soft MoE · RouteRank.

Mỗi bộ (CUB / Cars / In-Shop / SOP) = **một model riêng**, giao thức zero-shot chuẩn.

> Đổi bộ data: chỉ sửa **một biến `DATASET`** ở Mục 3, mọi cell phía dưới tự chạy theo.

## 1. Cài đặt

In [ ]:
# Cài các thư viện còn thiếu (Kaggle/Colab cần Internet ON).
# torch/torchvision đã có sẵn trên Kaggle & Colab; phần dưới chỉ bổ sung cái thiếu.
!pip install -q pytorch-metric-learning
import torch; print('CUDA:', torch.cuda.is_available())

## 2. Lấy code (tự clone từ GitHub nếu chưa có)
Chạy được cả khi mở từ repo đã clone (local) lẫn môi trường trống (Colab).

In [ ]:
# ── Lấy code: tự clone repo nếu chưa có (Colab/máy mới); bỏ qua nếu đã có sẵn ──
import os, sys

REPO_URL = 'https://github.com/trong5nhan6/retrieval.git'
REPO_DIR = 'retrieval'      # tên thư mục sau khi clone

if os.path.exists('config.py'):                       # đang ở ngay trong repo
    PROJECT_ROOT = os.path.abspath('.')
elif os.path.exists(os.path.join('..', 'config.py')): # đang ở notebooks/ (clone local)
    PROJECT_ROOT = os.path.abspath('..')
else:                                                 # môi trường trống -> clone
    if not os.path.isdir(REPO_DIR):
        os.system(f'git clone {REPO_URL} {REPO_DIR}')
    PROJECT_ROOT = os.path.abspath(REPO_DIR)

os.chdir(PROJECT_ROOT)                # cwd = gốc project (để train.py chạy thẳng)
sys.path.insert(0, PROJECT_ROOT)
print('PROJECT_ROOT =', PROJECT_ROOT)

## 3. Chọn bộ data + tải từ Drive
**Chỉ cần sửa biến `DATASET`.** Bộ nào có link Drive sẽ tự tải `.zip` về `datasets/` và giải nén.

In [ ]:
from config import HCFG

# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  CHỌN BỘ DATA — chỉ sửa duy nhất dòng này: 'cub' | 'cars' | 'inshop' | 'sop' ║
DATASET = 'cub'
# ╚══════════════════════════════════════════════════════════════════════════╝

# ── Link Google Drive cho từng bộ (.zip). Điền link rồi chọn DATASET tương ứng ─
DATASET_URLS = {
    'cub':    'https://drive.google.com/file/d/1bwP3KNX1Kaj-sPanx32KrCl4_NVM5m2P/view?usp=drive_link',
    'cars':   'https://drive.google.com/file/d/1zjDalOe1YFk6ire7KKaYe9RLqKBWbdpv/view?usp=drive_link',   # TODO: dán link Drive cho Cars196.zip
    'inshop': 'https://drive.google.com/file/d/1qz90vrI_F-QqkDmGqRU_4EyyGwwFQPja/view?usp=drive_link',   # TODO: dán link Drive cho In-shop.zip
    'sop':    'https://drive.google.com/file/d/1hA17Z6aJH99IkLDA-J-HVQk83xnecs14/view?usp=drive_link',   # TODO: dán link Drive cho Stanford_Online_Products.zip
}

DATASETS_DIR = os.path.join(PROJECT_ROOT, 'datasets')
os.makedirs(DATASETS_DIR, exist_ok=True)

import re, zipfile

def _drive_id(url):
    m = re.search(r'/d/([A-Za-z0-9_-]+)', url) or re.search(r'[?&]id=([A-Za-z0-9_-]+)', url)
    return m.group(1) if m else None

def fetch_dataset(name):
    """Tải <name>.zip từ Drive vào datasets/ rồi giải nén (bỏ qua nếu chưa có link)."""
    url = DATASET_URLS.get(name, '')
    if not url:
        print(f'[{name}] chưa có link Drive — bỏ qua (hãy điền DATASET_URLS).')
        return
    import gdown
    zip_path = os.path.join(DATASETS_DIR, f'{name}.zip')
    if not os.path.exists(zip_path):
        print(f'[{name}] tải từ Drive...')
        gdown.download(id=_drive_id(url), output=zip_path, quiet=False)
    print(f'[{name}] giải nén -> {DATASETS_DIR}')
    with zipfile.ZipFile(zip_path) as z:
        z.extractall(DATASETS_DIR)
    print(f'[{name}] xong.')

# Tải đúng bộ đang chọn
fetch_dataset(DATASET)

# ── Trỏ HCFG tới datasets/ (đường dẫn tuyệt đối) ─────────────────────────────
# SOP: trỏ tới thư mục ngoài; _parse_sop tự dò cấp lồng Stanford_Online_Products/ bên trong.
HCFG.data_roots = {
    'cub':    os.path.join(DATASETS_DIR, 'CUB_200_2011'),
    'cars':   os.path.join(DATASETS_DIR, 'Cars196'),
    'inshop': os.path.join(DATASETS_DIR, 'In-shop Clothes Retrieval Benchmark'),
    'sop':    os.path.join(DATASETS_DIR, 'Stanford_Online_Products'),
}
HCFG.epochs = 60
print('DATASET =', DATASET, '| root =', HCFG.data_roots[DATASET])

## 4. Chỉnh config trực tiếp (tuỳ chọn)
Sửa các tham số trong `OVERRIDES` để ghi đè `config.py` cho lần chạy này. Áp cho **cả** notebook (smoke-test/eval) **lẫn** `train.py` (qua file JSON). Để `OVERRIDES = {}` nếu dùng mặc định.

In [ ]:
# ── CHỈNH CONFIG TẠI ĐÂY: bỏ comment & sửa giá trị muốn đổi (so với config.py) ──
OVERRIDES = {
    # ── Training ──
    # 'epochs':          60,
    # 'frozen_epochs':   5,       # số epoch warmup (đóng băng backbone)
    # 'finetune_blocks': 2,       # số block ViT mở băng ở Stage-2 (0 = giữ đóng băng)
    # 'head_lr':         1e-4,
    # 'backbone_lr':     1e-5,
    # 'weight_decay':    1e-4,
    # 'grad_clip':       1.0,
    # 'eval_every':      5,        # đánh giá mỗi N epoch

    # ── Sampler / batch ──
    # 'classes_per_batch': 20,     # P lớp / batch
    # 'samples_per_class': 6,      # K ảnh / lớp  (batch_size ~= P*K)
    # 'batch_size':        128,

    # ── Loss ──
    # 'lambda_route':      0.5,    # trọng số routing-consistency (0 = tắt)
    # 'temperature':       0.07,
    # 'route_temperature': 0.1,

    # ── Soft MoE / kiến trúc ──
    # 'n_experts':        8,
    # 'slots_per_expert': 4,
    # 'expert_hidden':    512,
    # 'embed_dim':        256,
    # 'route_dim':        64,
    # 'tokens_per_stage': 64,
    # 'cnn_stages':       [0, 1, 2, 3],   # bỏ s1: [1, 2, 3]

    # ── RouteRank (inference) ──
    # 'rr_beta':    0.3,           # trọng số kênh routing khi fuse (0 = tắt RouteRank)
    # 'rr_topk':    10,
    # 'rr_alpha':   3.0,
    # 'rr_reroute': True,

    # ── Ablation switches (mới) ──
    # 'use_vit':  True,           # False -> tắt nhánh ViT
    # 'use_cnn':  True,           # False -> tắt nhánh CNN (ViT-only)
    # 'use_moe':  True,           # False -> bypass SoftMoE; tự tắt route loss + RouteRank

    # ── LR schedule (mới) ──
    # 'lr_schedule':   'cosine',  # 'cosine' | 'constant'
    # 'warmup_epochs': 0,         # số epoch warmup tuyến tính đầu Stage-1

    # ── Chọn loss chính trên z (mới) ──
    # 'loss_type':      'supcon', # 'supcon' | 'triplet' (khớp baseline triploss)
    # 'triplet_margin': 0.1,
    # 'triplet_miner':  True,     # đào semihard triplets

    # ── Embedding fusion (mới) ──
    # 'embed_dim':       512,      # chiều embedding (bỏ bottleneck 256)
    # 'use_cls_skip':    True,     # đường CLS -> embedding (sàn ≈ baseline)
    # 'local_gate_init': 0.0,      # γ khởi tạo nhánh MoE (0 => bắt đầu ≈ CLS)
    # 'bnneck':          True,     # BatchNorm trước L2 (False -> LayerNorm)
}

import json
# 1) áp cho HCFG đang dùng trong notebook (smoke-test & eval thấy ngay)
for k, v in OVERRIDES.items():
    if hasattr(HCFG, k):
        setattr(HCFG, k, v)
    else:
        print('Bỏ qua key không hợp lệ:', k)
# 2) ghi ra file để train.py (tiến trình con `!python`) đọc và áp y hệt
os.makedirs('results', exist_ok=True)
with open('results/_notebook_overrides.json', 'w', encoding='utf-8') as f:
    json.dump(OVERRIDES, f, ensure_ascii=False, indent=2)
print('Đã áp', len(OVERRIDES), 'override:', OVERRIDES or '(dùng mặc định config.py)')

## 5. Smoke-test shape (tải DINOv2 + ConvNeXt)

In [ ]:
from models.hybrid_encoder import HybridEncoder
from models.hyms_route import HyMSRoute
dev = 'cuda' if torch.cuda.is_available() else 'cpu'
enc = HybridEncoder(HCFG.vit_name, HCFG.cnn_name, dev,
                    use_vit=HCFG.use_vit, use_cnn=HCFG.use_cnn); enc.freeze_all()
m = HyMSRoute(enc, HCFG).to(dev)
z, rho, C = m(torch.randn(2, 3, 224, 224).to(dev))
print('z', z.shape,
      '| rho', None if rho is None else tuple(rho.shape),
      '| C',   None if C   is None else tuple(C.shape))
print('head params:', sum(p.numel() for p in m.head_parameters() if p.requires_grad))

## 6. Train bộ đang chọn

In [ ]:
!python train.py --dataset {DATASET} --seed 42

In [ ]:
# ── Xem biểu đồ đã auto-lưu trong results/ ───────────────────────────────────
from IPython.display import Image, display
run = f"hyms_{DATASET}_seed42"          # train.py dùng prefix 'hyms_' và seed 42
for p in (f'results/plot_test_{run}.png', f'results/plot_train_{run}.png'):
    if os.path.exists(p):
        print(p); display(Image(p))
    else:
        print('(chưa có)', p)

## 7. Eval lại từ checkpoint (base vs RouteRank)

In [ ]:
from data.dml_dataset import get_dml_loaders
from eval.routerank import evaluate_self, evaluate_query_gallery

L = get_dml_loaders(DATASET, HCFG)
ck = torch.load(f'results/checkpoints/best_hyms_{DATASET}_seed42.pt', map_location=dev)
m.load_state_dict(ck['model'])
rk = HCFG.recall_k_for(DATASET)
if DATASET == 'inshop':                       # query/gallery rời nhau
    r = evaluate_query_gallery(m, L['query'], L['gallery'], dev, HCFG, recall_k=rk)
else:                                          # CUB/Cars/SOP: self-retrieval trên test
    r = evaluate_self(m, L['test'], dev, HCFG, recall_k=rk)
print('BASE     :', r['base'])
print('RouteRank:', r.get('routerank', '(tắt — use_moe=False)'))

## Ablation (qua config)
`HCFG.cnn_stages=[1,2,3]` bỏ s1 · `HCFG.lambda_route=0` bỏ L_route · `HCFG.rr_beta=0` tắt RouteRank.